In [1]:
!nvidia-smi
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

Wed Oct 28 14:55:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q gdown
import gdown
url = 'https://drive.google.com/uc?id=1UOc0F36BJ3Y3gPNJ8WDZ8WvYnklCrL8N'
gdown.download(url, '/content/data.zip', quiet=False)
!rm -rf /content/extracted && mkdir -p /content/extracted
!unzip -q /content/data.zip -d /content/extracted
!ls -R /content/extracted | head -50

Downloading...
From (original): https://drive.google.com/uc?id=1UOc0F36BJ3Y3gPNJ8WDZ8WvYnklCrL8N
From (redirected): https://drive.google.com/uc?id=1UOc0F36BJ3Y3gPNJ8WDZ8WvYnklCrL8N&confirm=t&uuid=48dafe58-261a-4d8c-adf6-1cdd92aa1208
To: /content/data.zip
100%|██████████| 876M/876M [00:04<00:00, 202MB/s] 


/content/extracted:
__MACOSX
train+public_test

/content/extracted/__MACOSX:
train+public_test

/content/extracted/__MACOSX/train+public_test:
public_test
train

/content/extracted/__MACOSX/train+public_test/public_test:
audio
public_test_environment
zaic_tools

/content/extracted/__MACOSX/train+public_test/public_test/audio:
public_0002
public_0003
public_0004
public_0005
public_0006
public_0007
public_0008
public_0009
public_0010
public_0012
public_0015
public_0016
public_0017
public_0019
public_0020
public_0021
public_0022
public_0023
public_0024
public_0025
public_0026
public_0027
public_0028
public_0029
public_0030
public_0031
public_0032
public_0033
public_0034
public_0035
public_0036
public_0037
public_0038
ls: write error: Broken pipe


In [3]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes sentencepiece

In [4]:
import json, sys
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

import os
DATA_ROOT = None
for root, dirs, files in os.walk("/content/"):
    if "train+public_test" in dirs:
        DATA_ROOT = Path(root) / "train+public_test"
        break
if DATA_ROOT is None:
    DATA_ROOT = Path("/content/train+public_test")
print("DATA_ROOT =", DATA_ROOT)
SPLIT = "train"
MODEL_ID = "Qwen/Qwen3-1.7B"
OUTPUT_DIR = "/content/lifemate-qwen3-1.7b-lora"

DATA_ROOT = /content/extracted/train+public_test


In [5]:
# Load catalog
import json as _json
catalog = _json.load(open(DATA_ROOT / SPLIT / f"{SPLIT}_tools_catalog.json", encoding="utf-8"))
catalog_tools = {}
for app in catalog["apps"]:
    for t in app["tools"]:
        catalog_tools[t["name"]] = t

def tool_schema_text(installed):
    out = []
    for app_id, info in installed.items():
        for tool_name in info.get("tools", []):
            spec = catalog_tools.get(tool_name)
            if spec:
                out.append(_json.dumps(spec, ensure_ascii=False))
    return "\n".join(out)


In [6]:
SYSTEM = """You are a tool-using voice assistant agent.
You receive a Vietnamese user request, user context, conversation history, and available tools.
You must output ONLY a JSON array of tool calls to execute next.
Each item: {"name": "<app.tool>", "arguments": {<json args>}}.
Use only tools from the provided catalog. Read state before writes when IDs are unknown.
If information is missing, call sys.ask_missing_info with {"tool_name": "...", "params": ["..."]}.
If no installed app can handle the request, return [].
Do not include explanations or markdown. Output JSON only."""


In [7]:
rows = [json.loads(l) for l in open(DATA_ROOT / SPLIT / f"{SPLIT}.jsonl", encoding="utf-8") if l.strip()]
print("samples:", len(rows))

samples: 1545


In [8]:
def build_messages(item, executed_calls, executed_results, latest_text):
    installed = item["context"].get("installed_apps", {})
    schemas = tool_schema_text(installed)
    context = {
        "current_time": item["context"].get("current_time"),
        "timezone": item["context"].get("timezone"),
        "location": item["context"].get("location"),
        "user_profile": item["context"].get("user_profile"),
        "installed_apps": list(installed.keys()),
    }
    messages = [{"role": "system", "content": SYSTEM}]
    messages.append({"role": "user", "content": "Context:\n" + json.dumps(context, ensure_ascii=False) + "\n\nAvailable tool schemas:\n" + schemas})
    # history
    for h in item.get("history", []):
        if h.get("role") == "user":
            messages.append({"role": "user", "content": f"[User]: {h.get('audio_text','')}"})
        else:
            parts = []
            if h.get("text"):
                parts.append(f"Assistant: {h['text']}")
            if h.get("tool_calls"):
                parts.append("Tool calls: " + json.dumps(h["tool_calls"], ensure_ascii=False))
            if h.get("tool_results"):
                parts.append("Tool results: " + json.dumps(h["tool_results"], ensure_ascii=False))
            messages.append({"role": "assistant", "content": "\n".join(parts)})
    # executed current calls
    if executed_calls:
        messages.append({"role": "assistant", "content": "Tool calls: " + json.dumps(executed_calls, ensure_ascii=False)})
        messages.append({"role": "user", "content": "Tool results: " + json.dumps(executed_results, ensure_ascii=False)})
    messages.append({"role": "user", "content": f"Latest user request: {latest_text}\nReturn next tool calls as JSON array, or [] if done."})
    return messages


In [9]:
# Build SFT examples
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
examples = []
for item in rows:
    latest_text = item.get("audio_text", "")
    gold_calls = item.get("label", {}).get("tool_calls", [])
    gold_results = item.get("label", {}).get("tool_results", [])
    executed_calls = []
    executed_results = []
    # intermediate steps
    for k, call in enumerate(gold_calls):
        msgs = build_messages(item, executed_calls, executed_results, latest_text)
        target = [call]
        msgs_with_answer = msgs + [{"role": "assistant", "content": json.dumps(target, ensure_ascii=False)}]
        text = tok.apply_chat_template(msgs_with_answer, tokenize=False, add_generation_prompt=False)
        examples.append({"text": text})
        executed_calls.append(call)
        if k < len(gold_results):
            executed_results.append(gold_results[k])
        else:
            executed_results.append({"ok": True})
    # final done step
    msgs = build_messages(item, executed_calls, executed_results, latest_text)
    msgs_with_answer = msgs + [{"role": "assistant", "content": "[]"}]
    text = tok.apply_chat_template(msgs_with_answer, tokenize=False, add_generation_prompt=False)
    examples.append({"text": text})
    if not gold_calls:
        # ensure at least one [] example
        pass

print("examples:", len(examples))
dataset = Dataset.from_list(examples)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

examples: 5086


In [10]:
# QLoRA setup
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [11]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [14]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        save_steps=200,
        fp16=True,
        max_length=4096,
        gradient_checkpointing=True,
        max_grad_norm=0.0,
    ),
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)
print("saved to", OUTPUT_DIR)


Adding EOS to train dataset:   0%|          | 0/5086 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5086 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5086 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5086 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5086 [00:00<?, ? examples/s]

RuntimeError: gemm input type at::BFloat16 and output type float is only supported for CUDA devices with compute capability 8.0 or higher

In [ ]:
# Optional: merge adapter into base model for inference
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, trust_remote_code=True)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
merged.save_pretrained(OUTPUT_DIR + "-merged")
tok.save_pretrained(OUTPUT_DIR + "-merged")
print("merged saved")
